In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv("blinkit_orders.csv", encoding="latin-1")

In [10]:
# changing data types and creating calculated field  for (delivery delay in minutes) 

df = pd.read_csv("blinkit_orders.csv")
df['order_date'] = pd.to_datetime(df['order_date'])
df['promised_delivery_time'] = pd.to_datetime(df['promised_delivery_time'])
df['actual_delivery_time'] = pd.to_datetime(df['actual_delivery_time'])

# one simple calculated feature — delivery delay in minutes
df['delivery_delay_minutes'] = (df['actual_delivery_time'] - df['promised_delivery_time']).dt.total_seconds() / 60

X = df[['order_total', 'delivery_delay_minutes']].dropna()
print(X.head())
print(X.shape)

   order_total  delivery_delay_minutes
0      3197.07                    -5.0
1       976.55                     2.0
2       839.05                     4.0
3       440.23                    -1.0
4      2526.68                     1.0
(5000, 2)


C:\Users\HP\AppData\Local\Temp\ipykernel_8976\1998644731.py:4: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['order_date'] = pd.to_datetime(df['order_date'])
C:\Users\HP\AppData\Local\Temp\ipykernel_8976\1998644731.py:5: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['promised_delivery_time'] = pd.to_datetime(df['promised_delivery_time'])
C:\Users\HP\AppData\Local\Temp\ipykernel_8976\1998644731.py:6: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['actual_delivery_time'] = pd.to_datetime(df['actual_delivery_time'])


K MEANS CLUSTERING

In [5]:
pip install kneed

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
# THE ELBOW METHOD TO FIND THE K VALUE ( USING KNEE LOCATOR FOR AUTOMATIC PICKING UP THE K VALUE)

from sklearn.cluster import KMeans
from kneed import KneeLocator
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)   # X = your features dataframe

wcss = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_) 

kl = KneeLocator(range(1,11), wcss, curve='convex', direction='decreasing')
kl.elbow


np.int64(3)

In [18]:
#Making the k Means Clustering Model with the K value is 3

# create the model
model = KMeans(n_clusters=3, random_state=42)
labels = model.fit_predict(X_scaled)

#printing the final answer 
print("Cluster assigned to each point:", labels)
print("Centroid locations:", model.cluster_centers_)

# Attach labels back to your original data so you can interpret each cluster
df['cluster'] = labels
print(df.groupby('cluster').mean(numeric_only=True))


Cluster assigned to each point: [2 1 1 ... 2 1 1]
Centroid locations: [[-0.08389865  1.66839143]
 [-0.76400222 -0.41817522]
 [ 0.96537507 -0.37703103]]
             order_id   customer_id  order_total  delivery_partner_id  \
cluster                                                                 
0        5.011088e+09  5.121189e+07  2089.201698         51596.387164   
1        5.038570e+09  4.935438e+07  1207.612838         49825.319583   
2        5.027270e+09  5.040360e+07  3461.239464         49504.828228   

            store_id  delivery_delay_minutes  
cluster                                       
0        4928.003106               17.892340  
1        5046.744787                1.063010  
2        4980.785011                1.414661  


In [19]:
# Filter the DataFrame for only Cluster 0 (the delayed orders group)
cluster_0_df = df[df['cluster'] == 0]

# Check which stores appear most frequently in Cluster 0
print("Top stores contributing to Cluster 0 delays:")
print(cluster_0_df['store_id'].value_counts().head(5))

# Check which delivery partners appear most frequently in Cluster 0
print("\nTop delivery partners contributing to Cluster 0 delays:")
print(cluster_0_df['delivery_partner_id'].value_counts().head(5))

Top stores contributing to Cluster 0 delays:
store_id
6899    1
9809    1
9535    1
4187    1
9615    1
Name: count, dtype: int64

Top delivery partners contributing to Cluster 0 delays:
delivery_partner_id
73907    1
56077    1
79516    1
53868    1
88612    1
Name: count, dtype: int64
